# Galton Watson Process

Build a simple simulator of a Multi-type Galton-Watson process,
in which nodes are partitioned in C classes, and the number of class c'
 off-springs  of a class c  individual is given by a random variables Y^{(v)}_{c,c'}.

For  the case C=2,  consider the case in which Y^{(v)}_{c,c'}~Poisson(m_{c,c'}) with:
 
- m_{11}= 6/7*alpha
- m_{22}= 3/7*alpha
- m_{12}=m_{2,1}= 2/7* alpha        with alpha in {0.9, 0,95,  1.0, 1.05, 1.1},

and empirically evaluate the asymptotic extinction probability q.

(optional)   Extend the analysis to the a general multitype  Galton–Watson process with C=2. In particular investigate the conditions under which the process is supercritical.

### 1. The Core Task: Build a Simulator
You need to write a program that simulates a **Multi-type Galton-Watson process**.
* **Structure:** The population is divided into $C$ distinct classes.
* **Reproduction:** An individual of class $c$ produces a random number of offspring of class $c'$.
* **Notation:** The number of class $c'$ offspring produced by a specific individual $v$ of class $c$ is determined by the random variable $Y^{(v)}_{c,c'}$.

---

### 2. The Specific Experiment ($C=2$)
Once the simulator is built, you are asked to run a specific experiment with 2 classes ($C=2$).

**The Distribution**
The number of offspring follows a **Poisson distribution**:
$$Y^{(v)}_{c,c'} \sim \text{Poisson}(m_{c,c'})$$

**The Parameters**
The mean values ($m_{c,c'}$) for the Poisson distributions are defined by a scaling factor $\alpha$:
* **Class 1 producing Class 1:** $m_{11} = \frac{6}{7}\alpha$
* **Class 2 producing Class 2:** $m_{22} = \frac{3}{7}\alpha$
* **Cross-production (1 to 2 and 2 to 1):** $m_{12} = m_{21} = \frac{2}{7}\alpha$

**The Alpha Values**
You must run this simulation for the following values of $\alpha$:
$$\alpha \in \{0.9, 0.95, 1.0, 1.05, 1.1\}$$

**The Goal**
For each $\alpha$, you need to **empirically evaluate the asymptotic extinction probability ($q$)**.
> *Note: This means running the simulation many times for each $\alpha$ and counting how often the population eventually dies out (reaches 0 individuals).*

---

### 3. (Optional) General Extension
If you choose to do the optional part, you are asked to:
* Extend the analysis to a **general** Multi-type Galton-Watson process with $C=2$ (not limited to the specific Poisson means above).
* Investigate the theoretical conditions under which the process becomes **supercritical** (where there is a positive probability of survival forever).
 
> *Hint: This usually involves analyzing the spectral radius (largest eigenvalue) of the mean matrix $M = [m_{c,c'}]$.*

---

### Summary of Deliverables
1.  **Code:** A simulator for a multi-type branching process.
2.  **Data:** Empirical extinction probabilities ($q$) calculated for the 5 specific $\alpha$ values provided.
3.  **Analysis (Optional):** A study of the conditions for supercriticality in the general 2-type case.

In [13]:
# Import modules
import numpy as np
import matplotlib.pyplot as plt
import random
from enum import Enum
import tqdm

In [14]:
# -----------------------------------------------
# Population
# -----------------------------------------------
class IndividualClass(Enum):
    CLASS_A = 0
    CLASS_B = 1

class Individual:
    def __init__(self, ind_class: IndividualClass):
        self.ind_class = ind_class

    def reproduce(self, mean_matrix: np.ndarray) -> list:
        offspring = []
        if self.ind_class == IndividualClass.CLASS_A:
            num_offspring_A = np.random.poisson(mean_matrix[0][0])
            num_offspring_B = np.random.poisson(mean_matrix[0][1])
        else:
            num_offspring_A = np.random.poisson(mean_matrix[1][0])
            num_offspring_B = np.random.poisson(mean_matrix[1][1])
        
        offspring.extend([Individual(IndividualClass.CLASS_A) for _ in range(num_offspring_A)])
        offspring.extend([Individual(IndividualClass.CLASS_B) for _ in range(num_offspring_B)])
        return offspring

In [15]:
# -----------------------------------------------
# Galton-Watson Multi-type Tree
# -----------------------------------------------
class MultiTypeGW:
    def __init__(self, ancestor_class: IndividualClass, mean_matrix: np.ndarray):
        self.ancestor = Individual(ancestor_class)
        self.mean_matrix = mean_matrix
        self.current_generation = [self.ancestor]
        self.generations = []

    def step(self):
        next_generation = []
        for individual in self.current_generation:
            offspring = individual.reproduce(self.mean_matrix)
            next_generation.extend(offspring)
        self.generations.append(self.current_generation)
        self.current_generation = next_generation

In [24]:
# -----------------------------------------------
# Simulation Engine
# -----------------------------------------------
class SimulationEngine:
    def __init__(self, mean_matrix: np.ndarray, initial_class: IndividualClass, max_generations: int):
        self.mean_matrix = mean_matrix
        self.initial_class = initial_class
        self.max_generations = max_generations
        self.simulations_run = 0
        self.extinctions = 0
        self.last_generation = 0

    def run_simulation(self) -> bool:
        gw_process = MultiTypeGW(self.initial_class, self.mean_matrix)
        self.simulations_run += 1
        for generation in range(self.max_generations):
            if not gw_process.current_generation:
                self.extinctions += 1
                self.last_generation = generation
                return True  # Extinction
            gw_process.step()
        self.last_generation = self.max_generations
        return False  # Survived
    
    def estimate_extinction_probability(self) -> float:
        return self.extinctions / self.simulations_run if self.simulations_run > 0 else 0.0

In [25]:
# -----------------------------------------------
# Main Execution
# -----------------------------------------------
if __name__ == "__main__":
    np.random.seed(0)
    alpha_values = [0.9, 0.95, 1.0, 1.05, 1.1]
    mean_matrix_coefficients = [[6/7, 2/7], [2/7, 3/7]]
    MAX_GENERATIONS = 100
    NUM_SIMULATIONS = 100
    INITIAL_CLASS = IndividualClass.CLASS_A

    for alpha in alpha_values:
        mean_matrix = np.array(mean_matrix_coefficients) * alpha
        engine = SimulationEngine(mean_matrix, INITIAL_CLASS, MAX_GENERATIONS)
        avg_last_generation = 0
        
        print(f"Running simulations for alpha = {alpha}...")
        for _ in tqdm.tqdm(range(NUM_SIMULATIONS), desc=f"Simulations for alpha={alpha}", unit="sim"):
            engine.run_simulation()
            avg_last_generation += engine.last_generation
        
        avg_last_generation = avg_last_generation / NUM_SIMULATIONS
        extinction_prob = engine.estimate_extinction_probability()
        print(f"Alpha: {alpha}, Estimated Extinction Probability: {extinction_prob}, Average Last Generation: {avg_last_generation}")

Running simulations for alpha = 0.9...


Simulations for alpha=0.9: 100%|██████████| 100/100 [00:00<00:00, 11405.93sim/s]


Alpha: 0.9, Estimated Extinction Probability: 1.0, Average Last Generation: 5.03
Running simulations for alpha = 0.95...


Simulations for alpha=0.95: 100%|██████████| 100/100 [00:00<?, ?sim/s]


Alpha: 0.95, Estimated Extinction Probability: 1.0, Average Last Generation: 4.68
Running simulations for alpha = 1.0...


...

Alpha: 1.0, Estimated Extinction Probability: 0.98, Average Last Generation: 9.2
Running simulations for alpha = 1.05...


Simulations for alpha=1.05: 100%|██████████| 100/100 [00:01<00:00, 84.03sim/s]


Alpha: 1.05, Estimated Extinction Probability: 0.89, Average Last Generation: 15.49
Running simulations for alpha = 1.1...


Simulations for alpha=1.1: 100%|██████████| 100/100 [01:33<00:00,  1.07sim/s]

Alpha: 1.1, Estimated Extinction Probability: 0.75, Average Last Generation: 28.25
